# 01. Data Cleaning

**Data:** Kaggle [100,000 UK Used Car Data set](https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes) (UK used car listings, 2020)

- The 9 brand files (Audi, BMW, Ford, Hyundai, Mercedes, Skoda, Toyota, Vauxhall, VW) are combined into one table.
- `cclass.csv` and `focus.csv` are excluded because they are subsets of `merc.csv` and `ford.csv`; the `unclean *.csv` files are excluded because they are uncleaned duplicates.
- The cleaning rules and derived variables are defined as functions in [`src/features.py`](../src/features.py).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from features import load_raw, clean, pct_effect, PROCESSED_DIR, ROOT
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.3f}".format)

In [2]:
raw = load_raw()
print(raw.shape)
raw.head()

(99187, 10)


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,brand
0,A1,2017,12500,Manual,15735,Petrol,150,55.400,1.400,Audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.200,2.000,Audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.400,1.400,Audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.300,2.000,Audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.600,1.000,Audi


In [3]:
raw.isna().sum()

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
brand           0
dtype: int64

In [4]:
raw[["year", "price", "mileage"]].describe()

,year,price,mileage
count,"99,187.000","99,187.000","99,187.000"
mean,"2,017.088","16,805.348","23,058.914"
std,2.124,"9,866.773","21,148.524"
min,"1,970.000",450.000,1.000
25%,"2,016.000","9,999.000","7,425.000"
50%,"2,017.000","14,495.000","17,460.000"
75%,"2,019.000","20,870.000","32,339.000"
max,"2,060.000","159,999.000","323,000.000"


The summary shows impossible values (a maximum `year` of 2060) and should be checked for zero prices or mileage. Cleaning steps, in order:

1. Drop exact duplicate rows (repeated listings)
2. Keep model years 1990–2020 (removes typos such as 2060)
3. Drop rows with price or mileage of 0 or less
4. Trim the top and bottom 0.5% of prices (extreme values)
5. Trim the top 0.5% of annual mileage

**Derived variables**
- `age = 2020 − year` (listings were collected in 2020)
- `annual_mileage = mileage / max(age, 1)` (a 2020 car is under one year old, so the denominator is set to 1)
- `log_price = ln(price)`, so that a coefficient β can be read as a `(e^β − 1) × 100` percent change in price

In [5]:
df, log = clean(raw)
log

,step,rows
0,raw,99187
1,drop exact duplicates,97712
2,drop missing key fields,97712
3,year in 1990-2020,97709
4,price > 0 and mileage > 0,97709
5,trim price to 0.5-99.5 percentile,96739
6,trim annual mileage at 99.5 percentile,96255


In [6]:
df[["price", "age", "mileage", "annual_mileage"]].describe()

,price,age,mileage,annual_mileage
count,"96,255.000","96,255.000","96,255.000","96,255.000"
mean,"16,576.247",2.892,"22,653.036","7,382.609"
std,"8,881.213",1.980,"19,771.727","4,369.098"
min,"2,895.000",0.000,1.000,0.167
25%,"10,000.000",1.000,"7,693.000","4,313.000"
50%,"14,490.000",3.000,"17,528.000","6,800.000"
75%,"20,671.500",4.000,"32,073.500","9,870.000"
max,"59,450.000",24.000,"190,000.000","24,448.000"


In [7]:
df["brand"].value_counts()

brand
Ford          17585
Volkswagen    14768
Vauxhall      13065
Mercedes      12580
BMW           10408
Audi          10381
Toyota         6620
Skoda          6115
Hyundai        4733
Name: count, dtype: int64

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DIR / "cars_clean.csv", index=False)
print("saved", len(df), "rows")

saved 96255 rows
